# 04 Volatility (RQ3)

The volatility models of Chapter 4 and their results in Chapter 5.

1. Variance ratio around the announcement, regressed on an EU dummy and $\hat\sigma_i$.
2. GARCH(1,1) per firm, then a panel regression of the normalised excess variance
   $z^2_{it} = \hat\varepsilon^2_{it}/\hat\sigma^2_{it} - 1$ on the announcement indicator $A_{it}$ and $A_{it}\times EU_i$.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm
from arch import arch_model
from linearmodels import PanelOLS
from scipy.special import digamma

from src.common import TABLES, window_cols, star, clustered_ols, coef_table, load_sample, return_series

df = load_sample()
print(f"{len(df)} events")

615 events


## Variance ratio

- $VR_i = \sum_{k=-1}^{1} AR_{ik}^2 / (3\hat\sigma_i^2)$. Under no excess variance $3\,VR \sim \chi^2_3$:
  null median 0.789, null mean of $\log VR$ = $\psi(1.5) + \log 2 - \log 3 = -0.369$.

All regressions clustered by firm and calendar date.

In [2]:
def fit(y, X, z):
    firm = z["Region"].astype(str) + "_" + z["Identifier"].astype(int).astype(str)
    return clustered_ols(y.astype(float), X, z["Ticker"], z["Date"], firm, z["Date"])


rows = []

z2 = df.dropna(subset=window_cols(-1, 1) + ["Sigma"])
z2 = z2.assign(var_ratio=((z2[window_cols(-1, 1)] / 100) ** 2).sum(axis=1) / (z2["Sigma"] ** 2 * 3))
vr_reg = coef_table(fit(z2["var_ratio"], sm.add_constant(z2[["EU", "Sigma"]].astype(float)), z2))
vr_reg["variable"] = "VR:" + vr_reg["variable"]
rows.append(vr_reg)

VR_NULL_MEDIAN = float(st.chi2.median(3) / 3)
LOGVR_NULL_MEAN = float(digamma(1.5) + np.log(2) - np.log(3))
summary = []
for g in ("Full sample", "US", "EU"):
    zg = z2 if g == "Full sample" else z2[z2["Region"] == g]
    vr = zg["var_ratio"]
    const = pd.DataFrame({"const": 1.0}, index=vr.index)
    t_, p_ = st.ttest_1samp(vr, 1.0)
    mc = fit(vr - 1.0, const, zg)
    summary.append(dict(variable=f"mean VR {g}", coef=float(vr.mean()), se=np.nan,
                        t=float(t_), p=float(p_), sig=star(float(p_)), N=len(vr),
                        median=float(vr.median()), skew=float(vr.skew()),
                        t_2way=float(mc.tstats["const"]), p_2way=float(mc.pvalues["const"])))
    lvr = np.log(vr)
    ml = fit(lvr - LOGVR_NULL_MEAN, const, zg)
    n_below = int((vr < VR_NULL_MEDIAN).sum())
    summary.append(dict(variable=f"dist VR {g}", coef=float(vr.median()), se=np.nan, t=np.nan,
                        p=float(st.binomtest(n_below, len(vr), 0.5).pvalue), sig=np.nan, N=len(vr),
                        median=float(vr.median()), skew=float(vr.skew()),
                        null_median=VR_NULL_MEDIAN, share_below_1=float((vr < 1).mean()),
                        share_below_null_median=(n_below / len(vr)),
                        mean_logVR=float(lvr.mean()), null_mean_logVR=LOGVR_NULL_MEAN,
                        t_2way_logVR=float(ml.tstats["const"]),
                        p_2way_logVR=float(ml.pvalues["const"]),
                        trimmed_mean_5pct=float(st.trim_mean(vr, 0.05)),
                        share_above_10=float((vr > 10).mean())))
rows.append(pd.DataFrame(summary))

# log VR: checks whether the EU coefficient comes from the bulk of the distribution or the tail
z2 = z2.assign(log_var_ratio=np.log(z2["var_ratio"]))
m_log = fit(z2["log_var_ratio"], sm.add_constant(z2[["EU", "Sigma"]].astype(float)), z2)
log_reg = coef_table(m_log)
log_reg["variable"] = "logVR:" + log_reg["variable"]
rows.append(log_reg)
log_eu = coef_table(fit(z2["log_var_ratio"], sm.add_constant(z2[["EU"]].astype(float)), z2))
log_eu = log_eu[log_eu["variable"] == "EU"].assign(variable="logVR_EUonly:EU")
rows.append(log_eu)
# fitted E[log VR] at the mean sigma, comparable with the null of -0.369
sig_mean = float(z2["Sigma"].mean())
rows.append(pd.DataFrame([dict(variable="logVR fitted at mean Sigma",
                               coef=float(m_log.params["const"] + m_log.params["Sigma"] * sig_mean),
                               se=np.nan, t=np.nan, p=np.nan, sig=np.nan, N=int(m_log.nobs),
                               null_mean_logVR=LOGVR_NULL_MEAN, mean_Sigma=sig_mean)]))
rv_table = pd.concat(rows, ignore_index=True)
rv_table.to_csv(TABLES / "rv_regression.csv", index=False)
rv_table[["variable", "coef", "se", "t", "p", "N", "median", "t_2way", "p_2way"]]

,variable,coef,se,t,p,N,median,t_2way,p_2way
0,VR:const,1.5510,0.2123,7.3055,8.6509e-13,615,NaN,NaN,NaN
1,VR:EU,-0.1128,0.1760,-0.6407,5.2193e-01,615,NaN,NaN,NaN
2,VR:Sigma,-19.2993,13.9768,-1.3808,1.6784e-01,615,NaN,NaN,NaN
3,mean VR Full sample,1.2425,NaN,3.1734,1.5817e-03,615,0.6257,3.5993,0.0003
4,dist VR Full sample,0.6257,NaN,NaN,3.7560e-05,615,0.6257,NaN,NaN
5,mean VR US,1.2535,NaN,3.0408,2.4728e-03,545,0.6257,3.4308,0.0006
6,dist VR US,0.6257,NaN,NaN,7.8251e-05,545,0.6257,NaN,NaN
7,mean VR EU,1.1569,NaN,0.9085,3.6675e-01,70,0.6328,1.0219,0.3104
8,dist VR EU,0.6328,NaN,NaN,2.8198e-01,70,0.6328,NaN,NaN
9,logVR:const,-0.1158,0.1297,-0.8926,3.7243e-01,615,NaN,NaN,NaN


## GARCH(1,1)

Fitted per firm on its full daily return history (arch package). Firms whose fit does not
converge or has $a+b \ge 1$ are excluded. Announcement days shift to the next trading day
(at most 3 calendar days).

Panel (Equation 4.15): $z^2_{it} = \phi_1 A_{it} + \phi_2 (A_{it} \times EU_i) + \mu_i + \theta_t + u_{it}$, clustered by firm and date.
Main specification: normal innovations, day 0, untrimmed. Robustness: $a+b<0.99$,
Student-t innovations, window $[0,+1]$, symmetric 0.1%/99.9% trim.

In [3]:
series = return_series()
firm_events = df.groupby(["base", "Region", "Identifier"])["Date"].apply(set).to_dict()
MIN_OBS_GARCH = 200


def fit_garch(dist):
    out, summary, skipped = {}, [], {}
    for (base, region, ident), dates in firm_events.items():
        s = series[region].get(ident)
        if s is None:
            skipped["no_match"] = skipped.get("no_match", 0) + 1
            continue
        if len(s) < MIN_OBS_GARCH:
            skipped["too_short"] = skipped.get("too_short", 0) + 1
            continue
        try:
            res = arch_model(s * 100.0, vol="GARCH", p=1, q=1, dist=dist, rescale=False).fit(
                disp="off", show_warning=False)
        except Exception:
            skipped["garch_fail"] = skipped.get("garch_fail", 0) + 1
            continue
        if res.convergence_flag != 0:
            skipped["not_converged"] = skipped.get("not_converged", 0) + 1
            continue
        p = res.params
        persistence = float(p.get("alpha[1]", 0.0) + p.get("beta[1]", 0.0))
        if persistence >= 1.0:
            skipped["non_stationary"] = skipped.get("non_stationary", 0) + 1
            continue
        out[(base, region)] = (res.resid ** 2 / res.conditional_volatility ** 2 - 1.0, dates)
        summary.append(dict(firm=base, region=region, omega=float(p.get("omega", np.nan)),
                            alpha=float(p.get("alpha[1]", np.nan)), beta=float(p.get("beta[1]", np.nan)),
                            persistence=persistence, n_obs=len(s)))
    print(f"GARCH ({dist}): {len(out)} firms fitted, excluded {skipped}")
    return out, pd.DataFrame(summary), skipped


def build_panel(firm_z2, offsets=(0,), trim=False):
    parts = []
    for (base, region), (z2, dates) in firm_z2.items():
        a, hit = pd.Series(0.0, index=z2.index), 0
        for d0 in dates:
            d0 = pd.Timestamp(d0)
            if d0 not in a.index:
                nxt = a.index[a.index >= d0]
                if len(nxt) == 0 or (nxt[0] - d0).days > 3:
                    continue
                d0 = nxt[0]
            pos = a.index.get_loc(d0)
            for off in offsets:
                if 0 <= pos + off < len(a) and a.iloc[pos + off] == 0.0:
                    a.iloc[pos + off] = 1.0
                    hit += 1
        if hit:
            parts.append(pd.DataFrame({"firm": f"{base}|{region}", "date": z2.index, "z2": z2.values,
                                       "A": a.values, "EU": 1.0 if region == "EU" else 0.0}))
    panel = pd.concat(parts, ignore_index=True)
    panel["A_EU"] = panel["A"] * panel["EU"]
    if trim:
        lo, hi = panel["z2"].quantile([0.001, 0.999])
        panel = panel[(panel["z2"] >= lo) & (panel["z2"] <= hi)]
    return panel.set_index(["firm", "date"])


def z2_regression(panel):
    res = PanelOLS(panel["z2"], panel[["A", "A_EU"]], entity_effects=True, time_effects=True,
                   drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True, cluster_time=True)
    out = pd.DataFrame({"variable": res.params.index, "coef": res.params.values, "se": res.std_errors.values,
                        "t": res.tstats.values, "p": res.pvalues.values})
    out["sig"] = out["p"].apply(star)
    out["N"] = int(res.nobs)
    return out


z2_normal, garch_summary, skipped = fit_garch("normal")
garch_summary.to_csv(TABLES / "garch_summary.csv", index=False)
pd.DataFrame([{"reason": k, "n_firms": v} for k, v in sorted(skipped.items())]).to_csv(
    TABLES / "garch_exclusions.csv", index=False)
garch_summary.groupby("region")[["alpha", "beta", "persistence"]].median()

GARCH (normal): 170 firms fitted, excluded {'non_stationary': 4}


,alpha,beta,persistence
region,,,
EU,0.0639,0.9053,0.9756
US,0.0938,0.8404,0.9491


In [4]:
main = z2_regression(build_panel(z2_normal))
main.to_csv(TABLES / "garch_z2.csv", index=False)
main

,variable,coef,se,t,p,sig,N
0,A,0.0615,0.0966,0.6359,0.5249,,423317
1,A_EU,0.0085,0.2501,0.0340,0.9729,,423317


In [5]:
keep99 = set(zip(garch_summary.loc[garch_summary["persistence"] < 0.99, "firm"],
                 garch_summary.loc[garch_summary["persistence"] < 0.99, "region"]))
robust = {"persist99": build_panel({k: v for k, v in z2_normal.items() if k in keep99})}
z2_t, garch_summary_t, _ = fit_garch("t")
garch_summary_t.to_csv(TABLES / "garch_summary_tdist.csv", index=False)
robust["tdist"] = build_panel(z2_t)
robust["window01"] = build_panel(z2_normal, offsets=(0, 1))
robust["symtrim"] = build_panel(z2_normal, trim=True)

rows = []
for name, panel in robust.items():
    out = z2_regression(panel)
    out.to_csv(TABLES / f"garch_z2_{name}.csv", index=False)
    rows.append(out.assign(spec=name))
pd.concat([main.assign(spec="main")] + rows)[["spec", "variable", "coef", "se", "p", "N"]]

GARCH (t): 166 firms fitted, excluded {'non_stationary': 8}


,spec,variable,coef,se,p,N
0,main,A,0.0615,0.0966,0.5249,423317
1,main,A_EU,0.0085,0.2501,0.9729,423317
0,persist99,A,0.0904,0.1027,0.3783,364136
1,persist99,A_EU,0.1339,0.2596,0.6059,364136
0,tdist,A,0.1767,0.1442,0.2205,415008
1,tdist,A_EU,-0.1724,0.2769,0.5336,415008
0,window01,A,-0.0401,0.0684,0.5575,423317
1,window01,A_EU,0.0386,0.1808,0.8309,423317
0,symtrim,A,0.1221,0.0909,0.1794,422469
1,symtrim,A_EU,0.0666,0.2304,0.7727,422469
